# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *(número 29)* |
| **Integrantes** | · Dario Andres Guerrero Cantillo · |
| **Caso de estudio** | *(Wanderbricks- Clickstream)* |
| **Fecha de entrega** | domingo 30 de agosto |
| **🎥 Enlace al video** | https://drive.google.com/file/d/1JKq65AxOrN67qUIoS3CVEsmsIBlirZQb/view?usp=sharing |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema



Wanderbricks es una plataforma donde las personas pueden buscar y explorar propiedades para alquilar durante sus viajes. Cada vez que un usuario realiza una acción dentro de la página, como buscar un alojamiento, ver una propiedad, aplicar filtros o hacer clic en una opción, esa actividad queda registrada. Toda esta información permite conocer mejor cómo se comportan los usuarios mientras navegan por la plataforma.

El propósito de este trabajo es crear una base de datos analítica que permita almacenar y analizar esos registros de navegación de forma organizada. A medida que la cantidad de usuarios aumenta, también crece el volumen de información generada, por lo que es importante contar con una solución que facilite su procesamiento y análisis. Además, algunos datos contienen información agrupada en estructuras más complejas, por lo que se necesita una tecnología que pueda manejarlas sin dificultad.

Con esta base de datos se busca responder preguntas que pueden ayudar a mejorar la plataforma y apoyar la toma de decisiones. Por ejemplo, identificar qué acciones realizan más los usuarios, desde qué dispositivos navegan con mayor frecuencia, cuáles son las principales fuentes de tráfico y qué propiedades generan más interés. Conocer estos patrones permite entender mejor el comportamiento de los usuarios y encontrar oportunidades para mejorar su experiencia dentro de Wanderbricks.

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

database,tableName,isTemporary
wanderbricks,amenities,false
wanderbricks,booking_updates,false
wanderbricks,bookings,false
wanderbricks,clickstream,false
wanderbricks,countries,false
wanderbricks,customer_support_logs,false
wanderbricks,destinations,false
wanderbricks,employees,false
wanderbricks,hosts,false
wanderbricks,page_views,false


In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

amenities                                38
booking_updates                      83,068
bookings                             72,247
clickstream                         100,000
countries                               168
customer_support_logs                 1,900
destinations                             42
employees                            73,006
hosts                                19,384
page_views                          500,000
payments                             49,638
properties                           18,163
property_amenities                  118,108
property_images                      54,186
reviews                              99,793
users                               124,509


In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)

In [0]:
from pyspark.sql import functions as F

tablas = [
    "users",
    "hosts",
    "properties",
    "bookings",
    "payments",
    "reviews",
    "clickstream"
]

for nombre in tablas:
    df = spark.table(f"samples.wanderbricks.{nombre}")
    
    print("\n" + "=" * 70)
    print(f"TABLA: {nombre}")
    print("=" * 70)
    
    print("\nESQUEMA:")
    df.printSchema()
    
    total = df.count()
    print(f"\nTOTAL DE REGISTROS: {total:,}")
    
    print("\nNULOS POR COLUMNA:")
    
    for columna in df.columns:
        nulos = df.filter(F.col(columna).isNull()).count()
        porcentaje = (nulos / total) * 100
        
        print(
            f"{columna:25s} "
            f"Nulos: {nulos:8,} "
            f"({porcentaje:6.2f}%)"
        )
    
    print("\nCARDINALIDAD:")
    
    for columna in df.columns:
        distintos = df.select(
            F.countDistinct(columna)
        ).first()[0]
        
        print(
            f"{columna:25s} "
            f"Valores distintos: {distintos:,}"
        )


TABLA: users

ESQUEMA:
root
 |-- user_id: long (nullable = false)
 |-- email: string (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- user_type: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- is_business: boolean (nullable = true)
 |-- company_name: string (nullable = true)


TOTAL DE REGISTROS: 124,509

NULOS POR COLUMNA:
user_id                   Nulos:        0 (  0.00%)
email                     Nulos:      124 (  0.10%)
name                      Nulos:        0 (  0.00%)
country                   Nulos:        0 (  0.00%)
user_type                 Nulos:        0 (  0.00%)
created_at                Nulos:        0 (  0.00%)
is_business               Nulos:        0 (  0.00%)
company_name              Nulos:   61,967 ( 49.77%)

CARDINALIDAD:
user_id                   Valores distintos: 124,509
email                     Valores distintos: 94,711
name                      Valores distintos: 93,216
country  

In [0]:
# Comprobación de relaciones entre las principales tablas

print("1. Reservas sin usuario:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.bookings b
LEFT JOIN samples.wanderbricks.users u
    ON b.user_id = u.user_id
WHERE u.user_id IS NULL
""").show()

print("2. Reservas sin propiedad:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.bookings b
LEFT JOIN samples.wanderbricks.properties p
    ON b.property_id = p.property_id
WHERE p.property_id IS NULL
""").show()

print("3. Propiedades sin anfitrión:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.properties p
LEFT JOIN samples.wanderbricks.hosts h
    ON p.host_id = h.host_id
WHERE h.host_id IS NULL
""").show()

print("4. Pagos sin reserva:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.payments p
LEFT JOIN samples.wanderbricks.bookings b
    ON p.booking_id = b.booking_id
WHERE b.booking_id IS NULL
""").show()

print("5. Reseñas sin usuario:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.reviews r
LEFT JOIN samples.wanderbricks.users u
    ON r.user_id = u.user_id
WHERE u.user_id IS NULL
""").show()

print("6. Reseñas sin propiedad:")
spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.reviews r
LEFT JOIN samples.wanderbricks.properties p
    ON r.property_id = p.property_id
WHERE p.property_id IS NULL
""").show()

1. Reservas sin usuario:
+--------+
|cantidad|
+--------+
|       0|
+--------+

2. Reservas sin propiedad:
+--------+
|cantidad|
+--------+
|       0|
+--------+

3. Propiedades sin anfitrión:
+--------+
|cantidad|
+--------+
|       0|
+--------+

4. Pagos sin reserva:
+--------+
|cantidad|
+--------+
|       8|
+--------+

5. Reseñas sin usuario:
+--------+
|cantidad|
+--------+
|       0|
+--------+

6. Reseñas sin propiedad:
+--------+
|cantidad|
+--------+
|       0|
+--------+



In [0]:
# Exploración específica de la tabla seleccionada: clickstream

clickstream = spark.table("samples.wanderbricks.clickstream")

print("Cantidad de registros:", clickstream.count())
print("\nEsquema:")
clickstream.printSchema()

print("\nPrimeros registros:")
display(clickstream.limit(10))

Cantidad de registros: 100000

Esquema:
root
 |-- event: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- referrer: string (nullable = true)
 |-- property_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: long (nullable = true)


Primeros registros:


event,metadata,property_id,timestamp,user_id
view,"List(desktop, email)",5135,2025-09-04T23:26:54.935Z,16390
search,"List(desktop, direct)",7805,2025-09-04T23:26:54.935Z,3655
click,"List(mobile, direct)",17349,2025-09-04T23:26:54.935Z,100094
filter,"List(tablet, ad)",6128,2025-09-04T23:26:54.935Z,102187
search,"List(mobile, google)",3438,2025-09-04T23:26:54.935Z,40863
search,"List(mobile, email)",14475,2025-09-04T23:26:54.935Z,115938
search,"List(tablet, ad)",4788,2025-09-04T23:26:54.935Z,69029
view,"List(desktop, direct)",12099,2025-09-04T23:26:54.935Z,37389
view,"List(mobile, google)",6600,2025-09-04T23:26:54.935Z,534
click,"List(mobile, google)",13528,2025-09-04T23:26:54.935Z,107117


In [0]:
# 1. Distribución de eventos en clickstream

display(
    clickstream
    .groupBy("event")
    .count()
    .orderBy(F.desc("count"))
)

event,count
click,25216
filter,24961
search,24934
view,24889


In [0]:
# 2. Extraer los campos anidados de metadata

clickstream_metadata = clickstream.select(
    "event",
    "user_id",
    "property_id",
    "timestamp",
    F.col("metadata.device").alias("device"),
    F.col("metadata.referrer").alias("referrer")
)

display(clickstream_metadata.limit(20))

event,user_id,property_id,timestamp,device,referrer
view,16390,5135,2025-09-04T23:26:54.935Z,desktop,email
search,3655,7805,2025-09-04T23:26:54.935Z,desktop,direct
click,100094,17349,2025-09-04T23:26:54.935Z,mobile,direct
filter,102187,6128,2025-09-04T23:26:54.935Z,tablet,ad
search,40863,3438,2025-09-04T23:26:54.935Z,mobile,google
search,115938,14475,2025-09-04T23:26:54.935Z,mobile,email
search,69029,4788,2025-09-04T23:26:54.935Z,tablet,ad
view,37389,12099,2025-09-04T23:26:54.935Z,desktop,direct
view,534,6600,2025-09-04T23:26:54.935Z,mobile,google
click,107117,13528,2025-09-04T23:26:54.935Z,mobile,google


In [0]:
# 3. Distribución por dispositivo

display(
    clickstream_metadata
    .groupBy("device")
    .count()
    .orderBy(F.desc("count"))
)

device,count
tablet,33753
desktop,33303
mobile,32944


In [0]:
# 4. Distribución por fuente de referencia

display(
    clickstream_metadata
    .groupBy("referrer")
    .count()
    .orderBy(F.desc("count"))
)

referrer,count
email,25157
ad,24995
direct,24934
google,24914


In [0]:
# 5. Tipo de evento según dispositivo

display(
    clickstream_metadata
    .groupBy("device", "event")
    .count()
    .orderBy("device", F.desc("count"))
)

device,event,count
desktop,click,8437
desktop,filter,8351
desktop,view,8317
desktop,search,8198
mobile,click,8334
mobile,search,8244
mobile,view,8186
mobile,filter,8180
tablet,search,8492
tablet,click,8445


In [0]:
# 6. Tipo de evento según fuente de referencia

display(
    clickstream_metadata
    .groupBy("referrer", "event")
    .count()
    .orderBy("referrer", F.desc("count"))
)

referrer,event,count
ad,click,6320
ad,search,6243
ad,filter,6228
ad,view,6204
direct,search,6282
direct,filter,6237
direct,click,6218
direct,view,6197
email,view,6348
email,click,6339


In [0]:
# 7. Calidad de datos en clickstream

total = clickstream.count()

print(f"Total de registros: {total:,}")

for columna in ["event", "property_id", "timestamp", "user_id"]:
    nulos = clickstream.filter(F.col(columna).isNull()).count()
    porcentaje = (nulos / total) * 100
    
    print(
        f"{columna:15s} | "
        f"Nulos: {nulos:6,} | "
        f"Porcentaje: {porcentaje:.2f}%"
    )

print("\nNulos dentro de metadata:")

for columna in ["device", "referrer"]:
    nulos = clickstream_metadata.filter(F.col(columna).isNull()).count()
    porcentaje = (nulos / total) * 100
    
    print(
        f"{columna:15s} | "
        f"Nulos: {nulos:6,} | "
        f"Porcentaje: {porcentaje:.2f}%"
    )

Total de registros: 100,000
event           | Nulos:      0 | Porcentaje: 0.00%
property_id     | Nulos:      0 | Porcentaje: 0.00%
timestamp       | Nulos:      0 | Porcentaje: 0.00%
user_id         | Nulos:      0 | Porcentaje: 0.00%

Nulos dentro de metadata:
device          | Nulos:      0 | Porcentaje: 0.00%
referrer        | Nulos:      0 | Porcentaje: 0.00%


**Calidad de los datos que encuentro segun analisis**
Despues de evideciar el analisis de resultados nulos de la tabla clickstreams de sus columnas y de sus atributos en metadata los cuales dan como 100000 registros. tambien vemos que no hay datos nulos en event, property_id, timetamp como tampoco en user_id al igual vimos en los atributos que estan anidados que son metadata.divice y metadata.ferrer y tambien no hay datos nulos. esto quiere decir que no hay datos faltantes en los campos requeridos y vistos. esto es muy bueno porque nos facilita un poco mas el analisis mas adelante de otros puntos que encontramos en la actividad 1, igualmente falta analizar otras dimensiones muy importantes para poder pasar a las siguientes fases con los datos registrados hasta el momento que son cesarios para saber aun mas el comportamiento y movimientos de los usuarios de la taba clickstreams.


In [0]:
# 8. Comprobación de relaciones de clickstream

print("Eventos sin usuario correspondiente:")

spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.clickstream c
LEFT JOIN samples.wanderbricks.users u
    ON c.user_id = u.user_id
WHERE u.user_id IS NULL
""").show()


print("Eventos sin propiedad correspondiente:")

spark.sql("""
SELECT COUNT(*) AS cantidad
FROM samples.wanderbricks.clickstream c
LEFT JOIN samples.wanderbricks.properties p
    ON c.property_id = p.property_id
WHERE p.property_id IS NULL
""").show()

Eventos sin usuario correspondiente:
+--------+
|cantidad|
+--------+
|       0|
+--------+

Eventos sin propiedad correspondiente:
+--------+
|cantidad|
+--------+
|       0|
+--------+



**Relacion con los datos que tenemos**
comprobamos las relaciones de clickstreams con users y properties, utlizando una relacion de conexion con user_id y property id, lo cual nos reflejo que no se obtuvieron eventos sin propiedades correspondientes entre clickstreams y user_id al igual que clickstreams con property_ id, todo esto nos puede ayudar a saber que no hubo como tal datos huerfanos y que podemos tener relaciones entre la informacion de los users, como tambien con las properties con clickstreams para saber eventos de la navegacion y como tal hacer analisis mas cpmpletos mas adelante y poder entener un poco mas la informacion que queremos saber y estamos manejando.

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
|La estructura de los datos | se acomoda bien a las columnas que estan estructuradas, algo asi como ordenadas bien con la informacion con su tipo tales como las de clickstrams, las de user_id, la de las property_id, las de event y las de timestamp, que se miran bien estructuradas y se pueden entender de una forma mas facil diria, mas ordenada por datos.|se puede acomodar o manejar de una manera como muchos dicen y vemos flexibless y podemos tambien hacer cambios de informacion en los documentos, aunque en mi opinion, es una manera mas enriquecida de ordenar y estrucuturar los datos, siento que relacional es mejor que noSQL por la forma en como se escribe y se estrucutra los datos, pero las dos estrucutras son necesarias e importantes. |esta estructura diria que es la mejor opcion en cuanto a que permite trabajar con datos estructurados como semiestructurados desde el mismo entorno, a mi parecer es como una combinacion de relacional como tambien de NoSQL. | Lakehouse|
|Los Datos semiestructurados|Para manejar informacion aveces es importante hacer transformaciones para su analisis tales como las estructuras anidadas, demas informacion es normalmente|este paradigma si es el adecuado para las estructuras anidadas y tambien las flexibles, algo que en el paragima relacional se puede complicar un poco en su manejo por eso NoSQL es mejor opcion en este caso.|en este paragima de Lakehouse permite la conservasion o la presistencia para procesar directamente en apartados importantes de clickstream como lo es struct, en metadata con device y referrer|Lakehouse|
|Calidad y tranformacion de la informacion|se puede utilizar con claves y los join, por ejemplo con user_id y con property_id que nos dejan relacionar los eventos sin problemas.|es mas complicado porque se necesita un mayor tratamiento utlizando referencias y tambien depende del modelo que se este utilizando con esa informacion.|algo muy importante puede hacer en lakehouse y es hacer Join entre las tablas, al igual que puede conservar la flexibilidad de los datos semiestructurados que este utlizando, como tal lakehouse es un poco de las dos anteriores como anteriormente habia dicho.|Lakehouse|
|Manejo del volumen y de la escalabilidad|claramente puede manejar gran cantidad de volumenes aunque para hacer eso tiene que tener una adecuada escalabilidad a traves de su arquitectura relacional ante todo.|se caracteriza por su escalabilidad es de manera horizontal, algo diferente a la manera relacional.|Se distribuye con el procesamiento y el analisis de volumenes muy grandes, dado que utiliza Spark y Delta, por lo cual le permite hacer esto de manera tranquila, para el anailsis de grandes cantidades como anteriomente se habia dicho.|Lakehouse|

**Referencias (APA 7):**
Microsoft. (2026). ¿Qué es un almacén de lago de datos (Lakehouse)? Microsoft Learn. https://learn.microsoft.com/es-es/azure/databricks/lakehouse/


IBM. (2025). ¿Qué es Delta Lake? IBM Think. https://www.ibm.com/es-es/think/topics/delta-lake



---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo29"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

Trabajando en bigdata_grupo29.wanderbricks


### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

clickstream_schema = StructType([
    StructField("event", StringType(), True),
    StructField("user_id", LongType(), True),
    StructField("property_id", LongType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField(
        "metadata",
        StructType([
            StructField("device", StringType(), True),
            StructField("referrer", StringType(), True)
        ]),
        True
    )
])

print("Esquema explícito de clickstream creado correctamente.")

Esquema explícito de clickstream creado correctamente.


In [0]:
# 4.2.2. Ingesta de clickstream a la capa Bronze

clickstream_bronze = (
    spark.table("samples.wanderbricks.clickstream")
    .select(
        F.col("event").cast("string"),
        F.col("user_id").cast("long"),
        F.col("property_id").cast("long"),
        F.col("timestamp").cast("timestamp"),
        F.col("metadata").cast(clickstream_schema["metadata"].dataType)
    )
)

print("Datos preparados para la capa Bronze:")
print(f"Registros: {clickstream_bronze.count():,}")

clickstream_bronze.printSchema()

Datos preparados para la capa Bronze:
Registros: 100,000
root
 |-- event: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- property_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- device: string (nullable = true)
 |    |-- referrer: string (nullable = true)



In [0]:
# 4.2.3. Crear tabla Delta en la capa Bronze

tabla_bronze = "bigdata_grupo29.wanderbricks.bronze_clickstream"

(
    clickstream_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tabla_bronze)
)

print(f"Tabla Bronze creada correctamente: {tabla_bronze}")

Tabla Bronze creada correctamente: bigdata_grupo29.wanderbricks.bronze_clickstream


In [0]:
# 4.2.4. Verificación de la tabla Bronze

spark.sql("""
DESCRIBE TABLE EXTENDED bigdata_grupo29.wanderbricks.bronze_clickstream
""").show(truncate=False)

+----------------------------+--------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                 |comment|
+----------------------------+--------------------------------------------------------------------------+-------+
|event                       |string                                                                    |NULL   |
|user_id                     |bigint                                                                    |NULL   |
|property_id                 |bigint                                                                    |NULL   |
|timestamp                   |timestamp                                                                 |NULL   |
|metadata                    |struct<device:string,referrer:string>                                     |NULL   |
|                            |                                                          

**Ingesta Capa Bronce de los datos.**
se paso o se hizo la ingesta de datos que teniamos en Clickstream que se encontraba en el catalogo samples.wanderbricks hacia la capa bronce que teniamos creada propia que es bigdata_grupo29, despues de hacer la ingesta se dejaron los datos como estaban originalmente conservando su estructura como es, tales como los datos anidados de metadata como Struct que se componen como talde referrer y device. todo esto hace que creemos una nueva tabla que la llamamos **bigdata_grupo29.wanderbricks.bronze_clickstream** y lo importante e sque contiene los 100.000 registros almacenados que tienen una estrucutra de los tipos de datos que se tienen ahi los cuales son string, bigint, timestamp y struct. todo esto nos evidencia la capa delta que se tienen con las culmnas  al igual que los atributos anidados y metadatos.

### 4.3 Capa plata — datos limpios y tipados

In [0]:
# 4.3.1. Preparación de la capa Silver

clickstream_silver = (
    spark.table("bigdata_grupo29.wanderbricks.bronze_clickstream")
    .select(
        F.col("event").cast("string").alias("event"),
        F.col("user_id").cast("long").alias("user_id"),
        F.col("property_id").cast("long").alias("property_id"),
        F.col("timestamp").cast("timestamp").alias("timestamp"),
        F.col("metadata.device").cast("string").alias("device"),
        F.col("metadata.referrer").cast("string").alias("referrer")
    )
)

print(f"Registros preparados para Silver: {clickstream_silver.count():,}")

clickstream_silver.printSchema()

Registros preparados para Silver: 100,000
root
 |-- event: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- property_id: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- device: string (nullable = true)
 |-- referrer: string (nullable = true)



In [0]:
# 4.3.2. Crear tabla Delta en la capa Silver

tabla_silver = "bigdata_grupo29.wanderbricks.silver_clickstream"

(
    clickstream_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(tabla_silver)
)

print(f"Tabla Silver creada correctamente: {tabla_silver}")

Tabla Silver creada correctamente: bigdata_grupo29.wanderbricks.silver_clickstream


In [0]:
# 4.3.3. Verificación de la tabla Silver

spark.sql("""
DESCRIBE TABLE EXTENDED bigdata_grupo29.wanderbricks.silver_clickstream
""").show(truncate=False)

+----------------------------+---------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                  |comment|
+----------------------------+---------------------------------------------------------------------------+-------+
|event                       |string                                                                     |NULL   |
|user_id                     |bigint                                                                     |NULL   |
|property_id                 |bigint                                                                     |NULL   |
|timestamp                   |timestamp                                                                  |NULL   |
|device                      |string                                                                     |NULL   |
|referrer                    |string                                            

**Construimos la capa Plata**
con los datos de la tabla bronce, creamos la capa plata con la transformacion de los datos,  en este paso extragimos los atributos de device, referrer que se encontraban en  metadata, con todo esto teniendo en cuenta creamos la estrutura bigdata_grupo29.wanderbricks.silver_clickstream que contiene los 100.000 registros que se encontraban en la capa bronce creada anteriormente y que contiene event , user_id, property_id, timestamp, device y referrer. con la visualizacion de las tablas podemos ver la correcta creacion y administracion de l capa plata lo cual nos va a facilitar el analisis de los datos en esta etapa.

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas

In [0]:
# 4.4.1. Evidencia de estructura semiestructurada

spark.sql("""
SELECT
    event,
    user_id,
    property_id,
    timestamp,
    metadata
FROM bigdata_grupo29.wanderbricks.bronze_clickstream
LIMIT 10
""").show(truncate=False)

+------+-------+-----------+--------------------------+-----------------+
|event |user_id|property_id|timestamp                 |metadata         |
+------+-------+-----------+--------------------------+-----------------+
|view  |16390  |5135       |2025-09-04 23:26:54.935834|{desktop, email} |
|search|3655   |7805       |2025-09-04 23:26:54.935857|{desktop, direct}|
|click |100094 |17349      |2025-09-04 23:26:54.935865|{mobile, direct} |
|filter|102187 |6128       |2025-09-04 23:26:54.935872|{tablet, ad}     |
|search|40863  |3438       |2025-09-04 23:26:54.93588 |{mobile, google} |
|search|115938 |14475      |2025-09-04 23:26:54.935885|{mobile, email}  |
|search|69029  |4788       |2025-09-04 23:26:54.935891|{tablet, ad}     |
|view  |37389  |12099      |2025-09-04 23:26:54.935895|{desktop, direct}|
|view  |534    |6600       |2025-09-04 23:26:54.9359  |{mobile, google} |
|click |107117 |13528      |2025-09-04 23:26:54.935905|{mobile, google} |
+------+-------+-----------+----------

In [0]:
# 4.4.2. Acceso a los campos internos del STRUCT metadata

spark.sql("""
SELECT
    event,
    user_id,
    property_id,
    metadata.device AS device,
    metadata.referrer AS referrer
FROM bigdata_grupo29.wanderbricks.bronze_clickstream
LIMIT 10
""").show(truncate=False)

+------+-------+-----------+-------+--------+
|event |user_id|property_id|device |referrer|
+------+-------+-----------+-------+--------+
|view  |16390  |5135       |desktop|email   |
|search|3655   |7805       |desktop|direct  |
|click |100094 |17349      |mobile |direct  |
|filter|102187 |6128       |tablet |ad      |
|search|40863  |3438       |mobile |google  |
|search|115938 |14475      |mobile |email   |
|search|69029  |4788       |tablet |ad      |
|view  |37389  |12099      |desktop|direct  |
|view  |534    |6600       |mobile |google  |
|click |107117 |13528      |mobile |google  |
+------+-------+-----------+-------+--------+



**uso y manejo de los datos estructurados**
en la tabla clickstream tiene la informacion de datos semiestructurados que se tiene de metadata que contienen Struct que contiene los atributos device y referrer. despues de hacer unas consultas pudimos accder a indivialmente a los atributos de metadata.device y referrer con los cuales nos permite hacer un mejor tratamiento de los datos que se encontraban lo cual nos permite un analisis mas facil en capa plata, despues de haber tratado en la capa bronce.

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
# 5.1.1. Estado inicial de Silver

spark.sql("""
SELECT COUNT(*) AS total_registros
FROM bigdata_grupo29.wanderbricks.silver_clickstream
""").show()

+---------------+
|total_registros|
+---------------+
|         100000|
+---------------+



In [0]:
# 5.1.2. Operación controlada para evidenciar atomicidad

spark.sql("""
ALTER TABLE bigdata_grupo29.wanderbricks.silver_clickstream
ADD COLUMNS (prueba_atomicidad STRING)
""")

print("Transacción de modificación ejecutada correctamente.")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7117657058028059>, line 3
      1 # 5.1.2. Operación controlada para evidenciar atomicidad
----> 3 spark.sql("""
      4 ALTER TABLE bigdata_grupo29.wanderbricks.silver_clickstream
      5 ADD COLUMNS (prueba_atomicidad STRING)
      6 """)
      8 print("Transacción de modificación ejecutada correctamente.")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-pac

Esta celda fue ejecutada previamente para demostrar la evolución del esquema.
Si se ejecuta nuevamente, genera error porque la columna prueba_atomicidad ya existe.

In [0]:
# ACID — una operación que modifique datos
# TODO

In [0]:
# 5.1.3. Verificar la modificación realizada

spark.sql("""
DESCRIBE TABLE bigdata_grupo29.wanderbricks.silver_clickstream
""").show(truncate=False)

+-----------------+---------+-------+
|col_name         |data_type|comment|
+-----------------+---------+-------+
|event            |string   |NULL   |
|user_id          |bigint   |NULL   |
|property_id      |bigint   |NULL   |
|timestamp        |timestamp|NULL   |
|device           |string   |NULL   |
|referrer         |string   |NULL   |
|prueba_atomicidad|string   |NULL   |
+-----------------+---------+-------+



In [0]:
# 5.1.4. Historial de transacciones de la tabla Silver

spark.sql("""
DESCRIBE HISTORY bigdata_grupo29.wanderbricks.silver_clickstream
""").show(truncate=False)

+-------+-------------------+--------------+-------------------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                 |operation                        |operationParameters                                                                                                             

**Documentacion de Atomicidad**
se hizo en la estrucutra de la tabla agregando una nueva tabla para verificar la realizacion transaccional, por ende creamos una nueva tabla que la llamamos prueb atomicidad, anteriormente se tenia registro de 100000 registros que quedaron seguros sin ningun cambio y que se hizo una prueba aislada. todo esto nos da evidencia de un seguimiento a las modificaciones para tener un historial de transacciones.


In [0]:
# 5.1.5. Comprobar que los registros permanecen intactos

spark.sql("""
SELECT COUNT(*) AS total_registros
FROM bigdata_grupo29.wanderbricks.silver_clickstream
""").show()

+---------------+
|total_registros|
+---------------+
|         100000|
+---------------+



**Atomicidad y registro transaccional**

Se verificó  que la tabla silver_clickstream contenía 100.000 registros.

despues  se ejecutó una operación ALTER TABLE ADD COLUMNS, agregando la columna prueba_atomicidad. La modificación fue aplicada correctamente y quedó registrada en el historial de Delta Lake.

La consulta DESCRIBE HISTORY  mostró la creación de una nueva versión de la tabla  versión  2 con la operación ADD COLUMNS , evidenciando el registro transaccional de los cambios realizados.

como ultimo , se comprobó nuevamente que la tabla conserva los 100.000 registros originales, demostrando que la modificación del esquema no afectó la integridad de los datos.

In [0]:
# Time travel
# display(spark.sql(f"DESCRIBE HISTORY {TABLA}"))
# TODO: consultar una versión anterior y comparar

In [0]:
# 5.2.1. Consultar una versión anterior de la tabla

spark.sql("""
SELECT COUNT(*) AS total_registros
FROM bigdata_grupo29.wanderbricks.silver_clickstream VERSION AS OF 1
""").show()

**  consulta de versiones anteriores e historial**

Se utilizó la instrucción DESCRIBE HISTORY sobre la tabla silver_clickstream para visualizar el historial de cambios registrados por Delta Lake.

Posteriormente se realizó una consulta utilizando la característica de *time travel* mediante la cláusula VERSION AS OF. Se consultó la versión 1 de la tabla y se comprobó que contenía 100.000 registros.

Esta funcionalidad permite acceder al estado histórico de los datos y recuperar versiones anteriores de la tabla cuando sea necesario para auditoría, análisis o recuperación de información.

In [0]:
# Evolución de esquema con mergeSchema
# TODO

In [0]:
# 5.3.1. Mostrar el esquema actual

spark.sql("""
DESCRIBE TABLE bigdata_grupo29.wanderbricks.silver_clickstream
""").show(truncate=False)

**Evolución del esquema**

Para evidenciar y demostrar la evolución de esquema en Delta Lake se ejecutó una modificación mediante ALTER TABLE ADD COLUMNS , agregando la columna prueba_atomicidad a la tabla silver_clickstream.

despues se verificó la estructura de la tabla DESCRIBE TABLE, observándo la creacion de la nueva columna sin afectar los registros existentes.

Esta evidencia deja ver que Delta Lake deja evolucionar el esquema de una tabla de forma controlada, manteniendo la compatibilidad con los datos anteriormente almacenados.

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

In [0]:
# Consulta 1 — pregunta que responde:
# TODO

**¿Qué tipo de evento ocurre con mayor frecuencia?**

In [0]:
%sql
SELECT
    event,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY event
ORDER BY cantidad_eventos DESC;

event,cantidad_eventos
click,25216
filter,24961
search,24934
view,24889


In [0]:
from pyspark.sql.functions import count

df = spark.table("bigdata_grupo29.wanderbricks.silver_clickstream")

(df.groupBy("event")
   .agg(count("*").alias("cantidad_eventos"))
   .orderBy("cantidad_eventos", ascending=False)
   .show())

+------+----------------+
| event|cantidad_eventos|
+------+----------------+
| click|           25216|
|filter|           24961|
|search|           24934|
|  view|           24889|
+------+----------------+




**Resultado:** 
El evento más mas sucede en nuestro enfoque que clickstream es que  es click  contiene 25.216 ocurrencias, seguido por filter, search y view.

**Interpretación:**
 Los usuarios interactúan mas  activamente  con los elementos de la plataforma mediante clics. La distribución es muy  equilibrada o igualiatariamente entre los diferentes  tipos de eventos que se miran  que los usuarios realizan diversas acciones que hacen con su proceso de navegación, búsqueda y exploración de propiedades.

**Pregunta 2 ¿Qué dispositivos generan más actividad en la plataforma?**

In [0]:
%sql


SELECT
    device,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY device
ORDER BY cantidad_eventos DESC;

device,cantidad_eventos
tablet,33753
desktop,33303
mobile,32944


In [0]:
from pyspark.sql.functions import count

(df.groupBy("device")
   .agg(count("*").alias("cantidad_eventos"))
   .orderBy("cantidad_eventos", ascending=False)
   .show())


**Resultado:** 
Las tablets son las que hacen o generan  la mayor cantidad de eventos que son 33.753, despues siguen por desktop que son 33.303 y mobile  que son 32.944.

**Interpretación:** 
La actividad se encuentra repartida y organizada  de tal forma que es equilibrada entre los diferentes dispositivos. pero, las tablets tienen una ventaja en el volumen de interacciones, lo que quiere decir que las tablets tiene una experiencia de navegación favorable o buena para este tipo de dispositivo dentro de la plataforma.

**¿Qué fuentes de tráfico generan más actividad?**

In [0]:
%sql

SELECT
    referrer,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY referrer
ORDER BY cantidad_eventos DESC;

referrer,cantidad_eventos
email,25157
ad,24995
direct,24934
google,24914


In [0]:
from pyspark.sql.functions import count

(df.groupBy("referrer")
   .agg(count("*").alias("cantidad_eventos"))
   .orderBy("cantidad_eventos", ascending=False)
   .show())


**Resultado:** 
La fuente de tráfico con mucha mas  cantidad de eventos es el correo email con 25.157 eventos, seguida por ad, direct y google

**Interpretación:** 
Los eventos o recomendaciones del uso  de correo electrónico dejan ver y  generar un alto porcentaje y favoribilidad  la  de interacción de  de los usuarios. pero , las diferencias entre las diferentes fuentes son pequeñas, lo que deja ver una distribución equilibrada del tráfico de los diferentes canales de adquisición e informacion.

**¿Cuál es la combinación más frecuente entre dispositivo y tipo de evento?**

In [0]:
%sql

SELECT
    device,
    event,
    COUNT(*) AS cantidad_eventos
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY device, event
ORDER BY cantidad_eventos DESC
LIMIT 10;

device,event,cantidad_eventos
tablet,search,8492
tablet,click,8445
desktop,click,8437
tablet,filter,8430
tablet,view,8386
desktop,filter,8351
mobile,click,8334
desktop,view,8317
mobile,search,8244
desktop,search,8198



**Resultado:** 
La combinación que mas se repite es tablet y  search con 8.492 eventos,despues  sigue  tablet y  click  y como ultimo desktop + click.

**Interpretación:** 
Los usuarios que navegan desde las tablets muestran una muchas mas  actividad alta  de búsqueda e interacción. Esto quiere decir que estes dispositivo tiene una participación muy  importante en la  exploración de propiedades dentro de la plataforma.

In [0]:
from pyspark.sql.functions import count

(df.groupBy("device", "event")
   .agg(count("*").alias("cantidad_eventos"))
   .orderBy("cantidad_eventos", ascending=False)
   .show(10))

**¿Qué propiedades reciben más interacción por parte de los usuarios?**

In [0]:
%sql

SELECT
    property_id,
    COUNT(*) AS total_interacciones
FROM bigdata_grupo29.wanderbricks.silver_clickstream
GROUP BY property_id
ORDER BY total_interacciones DESC
LIMIT 10;

property_id,total_interacciones
4636,21
13172,18
13027,16
2151,16
16326,15
2224,15
3824,15
5445,15
4758,15
4865,15


In [0]:
from pyspark.sql.functions import count

(df.groupBy("property_id")
   .agg(count("*").alias("total_interacciones"))
   .orderBy("total_interacciones", ascending=False)
   .show(10))


**Resultado:** 
La propiedad 4636 deja ver el registro con  la mayor cantidad de interacciones con 21 eventos, despues  por las propiedades 13172 y 13027.

**Interpretación:** 
Estas propiedades hacen  una  mayor interés entre los usuarios y podrían ser  oportunidades comerciales importantes. Analizar sus características permitiría dejar ver  factores que incrementan la interacción y repetir  en otras propiedades de la plataforma.

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

**Resultados obtenidos de el analisis**

En esta parte de la actividad se realizaron varias pruebas sobre la tabla silver_clickstream para comprobar algunas de las funciones que ofrece Delta Lake dentro del entorno Lakehouse.

Primero se realizó una modificación en la tabla agregando una nueva columna llamada prueba_atomicidad.  Antes del cambio, la tabla tenía 100.000 registros y, después de realizar la modificación, se verificó que seguía teniendo la misma cantidad. Esto permitió comprobar que el cambio se aplicó correctamente sin afectar la información que ya estaba almacenada.

Luego se revisó el historial de la tabla utilizando el comando DESCRIBE HISTORY. Gracias a esto fue posible observar los cambios realizados y las diferentes versiones que se fueron generando. Después se consultó una versión anterior de la tabla utilizando VERSION AS OF, comprobando que los datos seguían disponibles y que la versión anterior también mantenía los 100.000 registros.

Por otra parte, se pudo evidenciar la evolución del esquema al observar que la nueva columna fue agregada correctamente a la estructura de la tabla. Lo importante es que este cambio se realizó sin tener que volver a crear la tabla ni cargar nuevamente los datos.

En general, los resultados obtenidos muestran que Delta Lake facilita el manejo de los datos, ya que permite realizar cambios de forma segura, mantener un historial de las modificaciones y consultar versiones anteriores cuando sea necesario. Estas características resultan muy útiles para proyectos de análisis de datos donde la información puede cambiar con el tiempo y es importante conservar un control sobre esos cambios.

---
## 6. Conclusiones




**Conclusiones de el analisis y como tal de la actividad realizada**

Durante el desarrollo de esta actividad se pudo comprobar que el enfoque Lakehouse resulta adecuado para trabajar con los datos de Wanderbricks, especialmente porque combina información estructurada y semiestructurada en un mismo entorno. En el caso de la tabla clickstream, fue posible trabajar tanto con los eventos de navegación como con la estructura anidada de metadata, sin necesidad de transformar completamente los datos desde el inicio.

Uno de los aspectos que funcionó mejor fue la construcción de las capas Bronze y Silver. La capa Bronze permitió conservar los datos originales tal como fueron recibidos, mientras que en la capa Silver se pudo preparar la información para el análisis, separando los campos de metadata en columnas más fáciles de consultar. Esto simplificó la creación de consultas analíticas tanto en SQL como en PySpark.

También fue posible comprobar algunas de las ventajas de Delta Lake, como el historial de cambios, la consulta de versiones anteriores y la evolución del esquema. Estas funcionalidades facilitaron el seguimiento de las modificaciones realizadas sobre la tabla y permitieron verificar que los datos se mantenían consistentes después de los cambios.

Durante la actividad surgieron algunas dificultades relacionadas con la gestión del esquema. Por ejemplo, después de agregar la columna prueba_atomicidad, no fue posible eliminarla directamente debido a restricciones de configuración de la tabla Delta. Sin embargo, esta situación permitió entender mejor cómo funciona la administración de esquemas y las limitaciones que pueden existir dependiendo de la configuración utilizada.

Si se tuviera que comenzar nuevamente, sería útil planificar desde el inicio las consultas de negocio que se desean responder y revisar con mayor detalle la configuración de las tablas Delta antes de realizar cambios en el esquema. Además, sería interesante incorporar más tablas del caso de estudio para realizar análisis que relacionen la navegación de los usuarios con reservas, pagos o reseñas, obteniendo así una visión más completa del comportamiento dentro de la plataforma.

En general, la actividad permitió comprender de manera práctica cómo implementar una solución analítica en Databricks utilizando una arquitectura Lakehouse y cómo aprovechar las capacidades de Delta Lake para gestionar y analizar datos de forma eficiente.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas